# 26. 组相对策略优化 Loss 源码解析与实现 (GRPO)

**难度：** Hard | **标签：** `微调`, `RLHF`, `GRPO`, `Loss Function` | **目标人群：** 模型微调与工程部署

**GRPO（Group Relative Policy Optimization）** 的思想就是：同一 prompt 下多条完整回复在组内比较奖励得到优势，从而省略 Critic；策略更新仍用 PPO 式的比率与裁剪。


### Step 1: 核心思想与痛点

> **on-policy 的问题：**
> 标准的 on-policy 的 RL (PPO) 需要训练 4 个模型：Actor（你要训练的模型）、Reference（参考模型，防止跑偏）、Reward Model（根据偏好训练的打分模型）和 Value Model（Critic）。训练非常不稳定，显存占用极大。
>
> **GRPO 的本质：** 对同一 prompt 并行采 $G$ 条**完整回复**，每条只有一个**标量奖励**；只在组内做均值方差标准化得到序列级 $\hat A_i$，再**广播到该回复的每个 token**，与逐 token 重要性比率 $r_t$ 相乘后套 **PPO Clip surrogate**——用**组内相对比较**代替逐 token 的 **GAE + Critic**，因此**不显式训练价值网络**（实现里仍可有 ref/KL 等正则，与本节主损失是两回事）。
>
> **与 DPO 的差别：** DPO 在**离线** chosen/rejected 成对偏好上，用 policy 与 ref 的隐式奖励差配合 `-log sigmoid` 做**偏好分类式**目标；本节仍是 **on-policy 风格的 PPO Actor clip**（比率 × 优势），数据形态与优化目标都不同。


### Step 2: GRPO 损失代码框架

本节代码里会用到三类张量：**组内标量奖励** `rewards_pg`，以及**逐 token 对数概率** `log_probs_new`（当前策略）与 `log_probs_old`（采样时的旧策略）。

1. 在 `rewards_pg` 的**最后一维**上求组内均值、标准差，做标准化，得到「每条完整回复一个」的序列级优势；再按与 `log_probs_*` 相同的 batch 展开顺序**展平**，并**扩一维**以便和序列长度广播对齐。
2. 逐 token 的重要性比率
3. 将序列级优势与上述比率按广播相乘后，套 **PPO 式 clip surrogate**：与 PPO 里的 `compute_actor_loss` 在数学形式上相同（`min` 未裁剪项与裁剪项，损失为负的均值）。


### Step 3: 核心公式

对每个 prompt，组内奖励 $R_1,\ldots,R_G$，均值 $\mu=\frac{1}{G}\sum_j R_j$；标准差 $\sigma$ 在最后一维上用 `torch.std(..., unbiased=False)` 计算。

1. **组相对优势（序列级）：** 若 $\sigma=0$，实现中对 `std` 做 `clamp_min(eps)`，等价于分母取 $\max(\sigma,\varepsilon)$：
   $$\hat{A}_i = \frac{R_i - \mu}{\max(\sigma,\ \varepsilon)}$$

2. **重要性采样比率 *(与 PPO 相同)*：** $r_t = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$，实现为逐 token 的 $\exp(\log\pi_\theta - \log\pi_{\text{old}})$

3. **Clip 截断目标 *(与 PPO 相同)*：** 将 $\hat{A}_i$ 广播到该序列每个位置后，surrogate 形式与第 12 节 Actor Clip 一致：
   $$L^{\text{CLIP}}(\theta) = - \mathbb{E}\big[\min(r_t \hat{A}_i,\ \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon)\,\hat{A}_i)\big]$$
   期望在 batch 与 token 上取平均；$\epsilon$ 为超参 `clip_range`（如 `0.2`）。


### Step 4: 动手实战

**要求**：请补全下方 `compute_grpo_loss` 函数。
为简化代码，假设前向阶段已得到组奖励 `rewards_pg` 与各 token 的 `log_probs_new` / `log_probs_old`。


In [ ]:
import torch


def compute_grpo_loss(
    rewards_pg: torch.Tensor,
    log_probs_new: torch.Tensor,
    log_probs_old: torch.Tensor,
    clip_range: float = 0.2,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    组相对策略优化 (GRPO)：组内奖励标准化得到序列级优势，再计算 PPO 式 Actor Clip 损失。

    Args:
        rewards_pg: 每个 prompt 下 G 条回复的标量奖励，形状 [num_prompts, group_size]
        log_probs_new: 当前策略在各 token 上的对数概率，形状 [num_prompts * group_size, seq_len]
        log_probs_old: 采样时旧策略，形状与 log_probs_new 相同
        clip_range: PPO 截断范围 ε，默认 0.2
        eps: 组内标准差下界，防止除零

    Returns:
        标量 loss（对 token 与 batch 取均值）
    """

    # ==========================================
    # TODO 1: 组相对优势（序列级）
    # 在 rewards_pg 最后一维上求 mean、std（unbiased=False，std 再 clamp_min(eps)），
    # 做 (rewards_pg - mean) / std 后 reshape 为 [num_prompts * group_size]
    # ==========================================
    # advantage_per_seq = ???

    # ==========================================
    # TODO 2: 重要性比率与广播
    # ratio = exp(log_probs_new - log_probs_old)；
    # adv = advantage_per_seq.unsqueeze(-1)，与 ratio 广播相乘
    # ==========================================
    # ratio = ???
    # adv = ???

    # ==========================================
    # TODO 3: 计算最终的 Loss
    # surr1 = ratio * adv
    # surr2 = torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range) * adv
    # loss = -min(surr1, surr2).mean()
    # ==========================================
    # loss = ???
    # return loss

    pass


In [ ]:
# 运行此单元格以测试你的实现
def test_compute_grpo_loss():
    try:
        torch.manual_seed(0)
        # 假设第一个 prompt 下三条回复质量不同(奖励 1/2/3), 第二个 prompt 下三条一样好(奖励全相同、组内方差为 0)
        rewards_pg = torch.tensor([[1.0, 2.0, 3.0], [10.0, 10.0, 10.0]])
        num_prompts, group_size = rewards_pg.shape
        seq_len = 2
        log_new = torch.randn(num_prompts * group_size, seq_len, requires_grad=True)
        log_old = torch.randn(num_prompts * group_size, seq_len)

        loss = compute_grpo_loss(rewards_pg, log_new, log_old, clip_range=0.2, eps=1e-8)

        assert loss.ndim == 0, "应返回标量 loss"

        mean_pg = rewards_pg.mean(dim=-1, keepdim=True)
        std_pg = rewards_pg.std(dim=-1, unbiased=False, keepdim=True).clamp_min(1e-8)
        adv_pg = (rewards_pg - mean_pg) / std_pg
        adv_flat = adv_pg.reshape(-1)
        ratio = torch.exp(log_new - log_old)
        adv = adv_flat.unsqueeze(-1)
        s1 = ratio * adv
        s2 = torch.clamp(ratio, 0.8, 1.2) * adv
        expected = -torch.min(s1, s2).mean()

        assert torch.allclose(loss, expected), f"Loss 计算错误: 期望 {expected}, 实际 {loss}"

        print("\n✅ All Tests Passed! GRPO 组相对优势 + PPO Clip 损失实现成功。")
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
    except TypeError as e:
        print("代码未完成，导致解包 None 错误")
    except Exception as e:
        print(f"\n❌ 测试失败: {e}")


test_compute_grpo_loss()



✅ All Tests Passed! GRPO 组相对优势 + PPO Clip 损失实现成功。


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


组相对策略优化用组内标量奖励标准化得到序列级优势，再接 PPO 的比率与 Clip，从而省掉 Critic。


In [ ]:
def compute_grpo_loss(
    rewards_pg: torch.Tensor,
    log_probs_new: torch.Tensor,
    log_probs_old: torch.Tensor,
    clip_range: float = 0.2,
    eps: float = 1e-8,
) -> torch.Tensor:
    # 1: 组相对优势（序列级）
    mean_pg = rewards_pg.mean(dim=-1, keepdim=True)
    std_pg = rewards_pg.std(dim=-1, unbiased=False, keepdim=True).clamp_min(eps)
    advantages_pg = (rewards_pg - mean_pg) / std_pg
    advantage_per_seq = advantages_pg.reshape(-1)

    # 2: 重要性比率与广播
    ratio = torch.exp(log_probs_new - log_probs_old)
    adv = advantage_per_seq.unsqueeze(-1)

    # 3: PPO Actor Clip
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range) * adv
    loss = -torch.min(surr1, surr2).mean()
    return loss
